# Part B1 — Instruction dataset (50 pairs)

Clinical Protocol Lookup Assistant (Variant 4), medical and clinical literature. Every pair is drafted from one page of the cleaned `domain_corpus/` text and carries the Variant 4 disclaimer. All logic lives in `src/instruction_dataset.py`; this notebook calls it and shows the results.

Pipeline: `chunks` (seeded page selection) → Claude drafts one pair per chunk from the committed prompt → `build` (per-pair checks, 50-pair selection, 40/10 split) → `validate` (all dataset checks plus token lengths).

In [1]:
# Dependencies come from requirements.txt, the single source of truth for every notebook in this repo.
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "instruction_dataset.py").is_file():
    ROOT = ROOT.parent
%pip install -q -r {ROOT}/requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, json
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
import instruction_dataset as d

pd.set_option("display.max_colwidth", 200, "display.width", 200)
chunks = d.read_jsonl(d.CHUNKS_PATH)
drafts = d.read_jsonl(d.DRAFTS_PATH)
meta = pd.DataFrame(d.read_jsonl(d.META_PATH))
rows = d.read_jsonl(d.SUBMISSION_PATH)
print(f"candidate chunks: {len(chunks)} (50 slots x primary + spare), saved drafts: {len(drafts)}, final pairs: {len(rows)}")
print("drafting log:")
print(json.dumps(json.loads(d.DRAFT_LOG_PATH.read_text(encoding="utf-8")), indent=2))

candidate chunks: 100 (50 slots x primary + spare), saved drafts: 51, final pairs: 50
drafting log:
{
  "drafter": "Claude (claude-sonnet-5-5) via Claude Code, one chunk at a time",
  "prompt_template": "prompts/instruction_draft_prompt.txt",
  "chunks": "data/instruction/chunks.jsonl (seed 42; rank 1 primary, rank 2 spare)",
  "raw_drafts": "data/instruction/drafts_raw.jsonl (saved drafts, one JSON object per attempt)",
  "validation_loop": "each draft was checked with check_draft() as it was written. Most failing drafts were reworded before being saved; one failing attempt (s10r1, attempt 1) is kept in drafts_raw.jsonl and was replaced by attempt 2.",
  "attempts_total": 51,
  "attempts_selected": 50,
  "attempts_failing_checks_kept_in_raw": 1,
  "spares_used": [
    "s08r2",
    "s17r2",
    "s23r2",
    "s25r2",
    "s28r2",
    "s42r2",
    "s45r2",
    "s48r2",
    "s50r2"
  ],
  "note": "Spare chunks replaced chunks that were tables of contents or evidence tables. Drafts avoid a

## Generation prompt template (exact text sent to the drafter)

The drafter is Claude (model id in `data/instruction/drafting_log.json`), not Mistral. The template follows the assignment's tip ("based ONLY on this text", `instruction` and `response` keys) and adds the locked sentence `Each response must be a concise 1–3 sentence answer in your own words. Do not copy text verbatim.` The placeholders `{pair_type}`, `{source_file}`, `{page}` and `{chunk_text}` are filled per chunk.

In [3]:
print(d.PROMPT_PATH.read_text(encoding="utf-8"))

You are drafting supervised fine-tuning data for a Clinical Protocol Lookup Assistant.

Read the text below and write exactly 1 instruction-response pair of type "{pair_type}" in JSON format, based ONLY on this text. The JSON object must have the keys "instruction" and "response".

Source document: {source_file}, page {page}.

Rules for the instruction:
- It is a question or request a clinician, nurse or pharmacist would ask on its own. It must make sense without the text and must not say "the text", "the passage" or "the page".
- It must match the requested type: definition (what is X), explanation (why or how X works), procedure (steps or recommended treatment), comparison (X versus Y), troubleshooting (what to do when something fails, is contraindicated or goes wrong), scenario (a short patient case and what the guideline advises), advantages_limitations (benefits and limits or cautions).
- Vary the wording. Do not start every instruction with the same words.

Rules for the response

## Five sample instruction–response pairs

One pair each from five different pair types and five different source families, printed straight from `instruction_dataset.jsonl`.

In [4]:
picked, seen_types, seen_fams = [], set(), set()
for m, row in zip(meta.to_dict("records"), rows):
    if m["type"] not in seen_types and m["family"] not in seen_fams:
        picked.append((m, row))
        seen_types.add(m["type"])
        seen_fams.add(m["family"])
    if len(picked) == 5:
        break
for m, row in picked:
    print(f"[{m['id']}] type={m['type']} | source={m['source_file']} page {m['page']}")
    print("INSTRUCTION:", row["instruction"])
    print("RESPONSE   :", row["response"])
    print()

[p01] type=procedure | source=who_malaria_2024-11-30.txt page 200
INSTRUCTION: How should primaquine be dosed to prevent relapse in uncomplicated P. vivax or P. ovale malaria?
RESPONSE   : Per WHO malaria guidelines, relapse prevention uses a high total primaquine dose of 7 mg/kg, delivered as 0.5 mg/kg daily over 14 days or as 1 mg/kg daily over 7 days. The 7-day schedule is reserved for patients with at least 70% G6PD activity, and G6PD status must be tested first, with pregnant women and infants under 1 month excluded. This output is for educational/reference purposes only and must not replace professional clinical judgment.

[p11] type=troubleshooting | source=cdc_sti_2021.txt page 99
INSTRUCTION: What should be done if a woman treated as an outpatient for pelvic inflammatory disease shows no improvement after 72 hours?
RESPONSE   : Per CDC STI treatment guidelines, clinical improvement is expected within 3 days of starting therapy, and if none appears within 72 hours of outpatient

## Validation

`run_validate()` re-runs every dataset check on the files on disk and stops on the first failure: 50 rows with only `instruction` and `response` keys; the locked type mix (definition 8, explanation 7, procedure 8, comparison 7, troubleshooting 6, scenario 7, advantages/limitations 7); exact disclaimer at the end of every response; 1–3 sentences and at least 30 clinical words before the disclaimer; no instruction template above 20% and at least two templates per type; no source family above 30%; every source file present in `domain_corpus/`; no 8-word run copied from the source page; every number in a response present in the source page; the source guideline named; no instruction near-duplicating the 3 fixed Part A prompts; the seed-42 split; and the spot-check file.

In [5]:
report = d.run_validate()

{
  "seed": 42,
  "pairs": 50,
  "train": 40,
  "eval": 10,
  "type_counts": {
    "procedure": 8,
    "troubleshooting": 6,
    "scenario": 7,
    "comparison": 7,
    "definition": 8,
    "advantages_limitations": 7,
    "explanation": 7
  },
  "family_counts": {
    "who_malaria": 8,
    "cdc_sti": 7,
    "icmr_antimicrobial": 7,
    "nice_sepsis": 5,
    "nice_hypertension": 5,
    "icmr_stw_vol1": 4,
    "icmr_stw_vol3": 4,
    "pubchem": 10
  },
  "template_counts": {
    "how should primaquine be": 1,
    "what single dose of": 1,
    "a child under <num>": 1,
    "when should school-based <term>": 1,
    "how should an antimalarial": 1,
    "in severe malaria which": 1,
    "what are non-immune travellers": 1,
    "what does <term> malaria": 1,
    "how is <term> genitalium": 1,
    "what penicillin regimen is": 1,
    "what should be done": 2,
    "how does <term> treatment": 1,
    "how do doxycycline and": 1,
    "a woman has persistent": 1,
    "how is hepatitis <term>": 1,

In [6]:
print("Counts by type:")
print(meta["type"].value_counts().to_string())
print()
print("Counts by source family (cap 30% = 15 pairs):")
print(meta["family"].value_counts().to_string())
print()
share = meta["template_family"].value_counts(normalize=True)
print(f"Instruction templates: {meta['template_family'].nunique()} distinct, largest share {share.iloc[0]:.0%} (limit 20%)")
print(f"Clinical words per response: mean {report['clinical_words_mean']}, min {report['clinical_words_min']}, max {report['clinical_words_max']}")
print("Tokens per example (Mistral tokenizer, instruction block + response + EOS):", report["tokens"])

Counts by type:
type
procedure                 8
definition                8
scenario                  7
comparison                7
advantages_limitations    7
explanation               7
troubleshooting           6

Counts by source family (cap 30% = 15 pairs):
family
pubchem               10
who_malaria            8
cdc_sti                7
icmr_antimicrobial     7
nice_sepsis            5
nice_hypertension      5
icmr_stw_vol1          4
icmr_stw_vol3          4

Instruction templates: 44 distinct, largest share 6% (limit 20%)
Clinical words per response: mean 68.5, min 46, max 91
Tokens per example (Mistral tokenizer, instruction block + response + EOS): {'tokenizer': 'mistralai/Mistral-7B-v0.1', 'mean_tokens': 173.5, 'p95_tokens': 219, 'max_tokens': 239, 'min_tokens': 132, 'total_tokens_one_epoch': 8674, 'proposed_max_seq_length': 256}


## Dataset size and token budget

- **50 pairs, 40 for training.** The aim is to teach the response shape (guideline named, 1–3 sentences, disclaimer, stop) and the habit of staying inside the source, not to add medical knowledge. Mistral-7B already knows the vocabulary; a small clean set avoids the over-fitting that a rank-32 adapter on a few dozen rows would show. Adapter B (r=16, alpha=32, `q_proj` and `v_proj`) is the balanced choice for this size.
- **Tokens.** The measured mean is about 174 tokens per example, p95 219 and max 239, so one epoch over the 40 training rows is roughly 7,000 tokens. A maximum sequence length of **256** covers every example without truncation, which is why B2 uses 256 instead of the 512 first proposed.
- **Compute.** At per-device batch 1 with gradient accumulation 4, three epochs are 30 optimizer steps. That fits easily in 4-bit on one RTX A6000 or A100.

In [7]:
trainable = 32 * 16 * ((4096 + 4096) + (4096 + 1024))  # Mistral-7B: 32 layers, q_proj 4096->4096, v_proj 4096->1024
print(f"Adapter B trainable parameters (r=16 on q_proj and v_proj): {trainable:,} = {trainable / 7_241_732_096:.3%} of the 7.24B base")

Adapter B trainable parameters (r=16 on q_proj and v_proj): 6,815,744 = 0.094% of the 7.24B base


## Manual spot-check (12 pairs, seeded sample)

The 12 ids come from `random.Random(42).sample` over the 50 ids, so the sample is reproducible. Each pair was read against its source page for correctness, grounding, the disclaimer and copying. The programmatic checks above already cover the disclaimer and copying for all 50 pairs.

In [8]:
spot = pd.read_csv(d.SPOTCHECK_PATH)
print(spot.drop(columns=["source_file"]).to_string(index=False))
print()
print("verdicts:", spot["verdict"].value_counts().to_dict())

 id                   type correct grounded disclaimer not_verbatim verdict                                                                                                                                            note
p02              procedure     yes      yes        yes          yes    pass                              Single 0.25 mg/kg primaquine dose, exclusions and the 25 to <50 kg dose of 7.5 mg match the WHO dosing table text.
p06             comparison     yes      yes        yes          yes    pass Antibiotics, no recommendation on exchange transfusion, phenobarbital 20 mg/kg mortality finding and corticosteroid advice all match the chunk.
p07             definition     yes      yes        yes          yes    pass             Definition of non-immune travellers, higher severe-malaria risk, >90% cure-rate rule and the chemoprophylaxis rule match the chunk.
p08 advantages_limitations     yes      yes        yes          yes    pass         Two non-randomized studies, 14-day 0

## Train / eval split (80/20, seed 42)

In [9]:
train, evals = d.read_jsonl(d.TRAIN_PATH), d.read_jsonl(d.EVAL_PATH)
print(f"train: {len(train)} examples, eval: {len(evals)} examples, seed {d.SEED}")
kind = dict(zip(meta["id"], meta["type"]))
print("train types:", pd.Series([kind[r["id"]] for r in train]).value_counts().to_dict())
print("eval types :", pd.Series([kind[r["id"]] for r in evals]).value_counts().to_dict())
print("overlap of ids:", {r["id"] for r in train} & {r["id"] for r in evals})

train: 40 examples, eval: 10 examples, seed 42
train types: {'definition': 8, 'procedure': 6, 'advantages_limitations': 6, 'troubleshooting': 5, 'comparison': 5, 'scenario': 5, 'explanation': 5}
eval types : {'scenario': 2, 'procedure': 2, 'comparison': 2, 'explanation': 2, 'troubleshooting': 1, 'advantages_limitations': 1}
overlap of ids: set()


## Inferences and limits

- **What the checks prove.** Format, counts, copying, number grounding and the disclaimer are verified for all 50 pairs by code, not by eye. The 12-pair spot-check adds a human reading of correctness against the source page.
- **What they do not prove.** A paraphrase can still drop a caveat (for example the hepatitis C screening exception for low-prevalence settings, noted in the spot-check). The pairs are drafted by an LLM, so they inherit its wording habits; template diversity and the 20% cap limit that, and the raw drafts are committed so the drafting can be audited.
- **Page choice.** Pages were chosen by seeded keyword fit to the pair type, not at random, to avoid tables of contents and evidence tables. Nine slots used the spare page because the primary was a contents page, an evidence table, or too close to the facts the three fixed prompts ask for.
- **Held-out prompts.** None of the 50 instructions is a near-duplicate of the 3 fixed Part A prompts, and the sepsis and hypertension pairs avoid the exact facts those prompts ask for, so Part B3 shows generalisation and not recall.
- **Eval split.** The 10 eval rows are too few for a stable loss estimate and happen to contain no `definition` pair; B3 therefore leans on the 3 fixed prompts and reports eval loss only as a rough signal.
- **Next step.** B2 trains Adapter B on `data/instruction/train.jsonl` with maximum sequence length 256, EOS appended to every text and loss on the response only.